# Univariate Analyse: Elo Rating Distribution

Deskriptive Statistik und Visualisierung der Elo-Ratings von Weiß und Schwarz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Lade Daten
df = pd.read_csv('games.csv')

# Erstelle kombiniertes Elo-Array
all_ratings = pd.concat([df['white_rating'], df['black_rating']], ignore_index=True)
df['avg_elo'] = (df['white_rating'] + df['black_rating']) / 2

print(f"Datensatz: {len(df)} Spiele")
print(f"Spieler insgesamt (White + Black): {len(all_ratings)}")
print(f"Eindeutige weiße Spieler: {df['white_id'].nunique()}")
print(f"Eindeutige schwarze Spieler: {df['black_id'].nunique()}")

In [ ]:
# Deskriptive Statistik
print("\n" + "="*70)
print("DESKRIPTIVE STATISTIK: ELO RATINGS")
print("="*70)

for name, data in [('White Rating', df['white_rating']), 
                    ('Black Rating', df['black_rating']),
                    ('Average Elo', df['avg_elo']),
                    ('All Ratings', all_ratings)]:
    print(f"\n{name}:")
    print(f"  Anzahl:      {len(data)}")
    print(f"  Mean:        {data.mean():.2f}")
    print(f"  Median:      {data.median():.2f}")
    print(f"  Std Dev:     {data.std():.2f}")
    print(f"  Minimum:     {data.min():.0f}")
    print(f"  Q1 (25%):    {data.quantile(0.25):.0f}")
    print(f"  Q3 (75%):    {data.quantile(0.75):.0f}")
    print(f"  Maximum:     {data.max():.0f}")
    print(f"  IQR:         {data.quantile(0.75) - data.quantile(0.25):.0f}")
    print(f"  Skewness:    {data.skew():.3f}")
    print(f"  Kurtosis:    {data.kurtosis():.3f}")

In [ ]:
# Histogramme mit KDE
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Elo Rating Verteilungen (Histogram + KDE)', fontsize=14, fontweight='bold')

datasets = [
    (df['white_rating'], axes[0, 0], 'White Rating'),
    (df['black_rating'], axes[0, 1], 'Black Rating'),
    (df['avg_elo'], axes[1, 0], 'Average Elo'),
    (all_ratings, axes[1, 1], 'All Ratings Combined')
]

for data, ax, title in datasets:
    ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black', density=True)
    
    # KDE hinzufügen
    data.plot(kind='kde', ax=ax, color='red', linewidth=2)
    
    ax.set_xlabel('Elo Rating')
    ax.set_ylabel('Häufigkeit (Dichte)')
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box Plots
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Elo Rating Box Plots - Ausreißer & Quartile', fontsize=14, fontweight='bold')

# White vs Black
data_to_plot = [df['white_rating'].dropna(), df['black_rating'].dropna()]
bp1 = axes[0].boxplot(data_to_plot, labels=['White', 'Black'], patch_artist=True)
for patch, color in zip(bp1['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
axes[0].set_ylabel('Elo Rating')
axes[0].set_title('White vs Black Ratings')
axes[0].grid(alpha=0.3, axis='y')

# Average Elo
bp2 = axes[1].boxplot([df['avg_elo'].dropna()], labels=['Avg Elo'], patch_artist=True)
bp2['boxes'][0].set_facecolor('lightgreen')
axes[1].set_ylabel('Elo Rating')
axes[1].set_title('Average Elo Rating')
axes[1].grid(alpha=0.3, axis='y')

# Violin Plot
parts = axes[2].violinplot([df['white_rating'].dropna(), df['black_rating'].dropna()], 
                           positions=[1, 2], showmeans=True, showmedians=True)
axes[2].set_xticks([1, 2])
axes[2].set_xticklabels(['White', 'Black'])
axes[2].set_ylabel('Elo Rating')
axes[2].set_title('Elo Distributions (Violin Plot)')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Q-Q Plots: Test auf Normalverteilung
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Q-Q Plots: Normalverteilungs-Test', fontsize=14, fontweight='bold')

datasets = [
    (df['white_rating'], axes[0, 0], 'White Rating'),
    (df['black_rating'], axes[0, 1], 'Black Rating'),
    (df['avg_elo'], axes[1, 0], 'Average Elo'),
    (all_ratings, axes[1, 1], 'All Ratings')
]

for data, ax, title in datasets:
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f'Q-Q Plot: {title}')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Normalitätstests
print("\n" + "="*70)
print("NORMALITÄTSTESTS (Shapiro-Wilk Test)")
print("="*70)
print("H0: Daten sind normalverteilt")
print("Wenn p-value < 0.05: Daten sind NICHT normalverteilt\n")

test_data = [
    ('White Rating', df['white_rating']),
    ('Black Rating', df['black_rating']),
    ('Average Elo', df['avg_elo'])
]

# Shapiro-Wilk Test (max 5000 samples)
for name, data in test_data:
    sample = data.dropna().sample(min(5000, len(data)), random_state=42)
    stat, p_value = stats.shapiro(sample)
    
    print(f"{name}:")
    print(f"  Test Statistic: {stat:.4f}")
    print(f"  P-value: {p_value:.2e}")
    
    if p_value > 0.05:
        print(f"  ✓ Normalverteilt (p > 0.05)\n")
    else:
        print(f"  ✗ NICHT normalverteilt (p < 0.05)\n")

# Additional: Anderson-Darling Test
print("\nAnderson-Darling Test (strikter):")
for name, data in test_data:
    result = stats.anderson(data.dropna())
    print(f"{name}: Statistic = {result.statistic:.4f}")
    if result.statistic > result.critical_values[2]:
        print(f"  ✗ NICHT normalverteilt (auf 5% Niveau)\n")
    else:
        print(f"  ✓ Normalverteilt (auf 5% Niveau)\n")

In [ ]:
# Ausreißer-Erkennung (IQR-Methode)
print("\n" + "="*70)
print("AUSREISSER-ANALYSE (IQR-Methode)")
print("="*70)
print("Ausreißer = Werte außerhalb [Q1 - 1.5*IQR, Q3 + 1.5*IQR]\n")

for name, data in [('White Rating', df['white_rating']), 
                     ('Black Rating', df['black_rating'])]:
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    print(f"{name}:")
    print(f"  Q1: {Q1:.0f}, Q3: {Q3:.0f}, IQR: {IQR:.0f}")
    print(f"  Lower Bound: {lower_bound:.0f}")
    print(f"  Upper Bound: {upper_bound:.0f}")
    print(f"  Anzahl Ausreißer: {len(outliers)} ({100*len(outliers)/len(data):.1f}%)")
    
    if len(outliers) > 0:
        print(f"  Ausreißer-Range: {outliers.min():.0f} - {outliers.max():.0f}")
    print()

In [ ]:
# Kumulative Verteilung (ECDF)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Empirische kumulative Verteilungsfunktion (ECDF)', fontsize=14, fontweight='bold')

# White vs Black
for data, ax, datasets_plot in [
    (None, axes[0], [('White', df['white_rating'].dropna(), 'blue'), 
                     ('Black', df['black_rating'].dropna(), 'red')]),
    (None, axes[1], [('All Players', all_ratings.dropna(), 'green')])
]:
    for label, data_sorted, color in datasets_plot:
        sorted_data = np.sort(data_sorted)
        y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        ax.plot(sorted_data, y, label=label, linewidth=2, color=color)
    
    ax.set_xlabel('Elo Rating')
    ax.set_ylabel('Kumulative Häufigkeit')
    ax.legend()
    ax.grid(alpha=0.3)

axes[0].set_title('White vs Black')
axes[1].set_title('Alle Spieler')

plt.tight_layout()
plt.show()

In [ ]:
# Zusammenfassung
print("\n" + "="*70)
print("ZUSAMMENFASSUNG: UNIVARIATE ELO-ANALYSE")
print("="*70)

print(f"\n📊 DATENSATZ")
print(f"   • Spiele: {len(df):,}")
print(f"   • White Spieler: {df['white_id'].nunique():,}")
print(f"   • Black Spieler: {df['black_id'].nunique():,}")

print(f"\n📈 ELO-VERTEILUNG")
print(f"   White:")
print(f"     - Mean: {df['white_rating'].mean():.0f}")
print(f"     - Median: {df['white_rating'].median():.0f}")
print(f"     - Std Dev: {df['white_rating'].std():.0f}")
print(f"     - Range: {df['white_rating'].min():.0f} - {df['white_rating'].max():.0f}")

print(f"\n   Black:")
print(f"     - Mean: {df['black_rating'].mean():.0f}")
print(f"     - Median: {df['black_rating'].median():.0f}")
print(f"     - Std Dev: {df['black_rating'].std():.0f}")
print(f"     - Range: {df['black_rating'].min():.0f} - {df['black_rating'].max():.0f}")

print(f"\n   Durchschnittlich:")
print(f"     - Mean: {df['avg_elo'].mean():.0f}")
print(f"     - Median: {df['avg_elo'].median():.0f}")
print(f"     - Std Dev: {df['avg_elo'].std():.0f}")

print(f"\n✓ ERKENNTNISSE")
skew_w = df['white_rating'].skew()
skew_b = df['black_rating'].skew()
if skew_w > 0 and skew_b > 0:
    print(f"   • Beide Verteilungen sind RECHTSCHIEF (positive skewness)")
    print(f"     → Mehr niedrig-bewertete Spieler als hoch-bewertete")
else:
    print(f"   • White Skewness: {skew_w:.3f}, Black Skewness: {skew_b:.3f}")

print(f"   • White und Black haben ähnliche Verteilungen")
print(f"   • Typische Elo-Werte: {df['avg_elo'].quantile(0.25):.0f} - {df['avg_elo'].quantile(0.75):.0f}")